# ⚡ HireSense AI
### RAG-Powered Talent Intelligence Platform

**Team:** Waleed El-Jack · Kristofor Figueiredo · Kevin Ordet · Jamie Shook  
**Course:** MAS 691 — Large Language Models | April 2026

---

Upload real resumes, paste or auto-fetch a job description from any URL, then run the full RAG pipeline to score and rank candidates with explainable AI output.


**Run cells top-to-bottom. Cell 5 launches the full interactive dashboard.**

## 📦 Cell 1 — Install & Setup

In [ ]:
!pip install openai anthropic scikit-learn numpy ipywidgets requests --quiet

from google.colab import output
output.enable_custom_widget_manager()

print('✅ All packages installed.')

## 🔑 Cell 2 — API Keys

**Option B (Colab Secrets) is recommended** — click the 🔑 key icon in the left sidebar.

In [ ]:
# ── Option A: Paste directly ─────────────────────────────────────────────────
OPENAI_API_KEY    = ''   # sk-...
ANTHROPIC_API_KEY = ''   # sk-ant-...

# ── Option B: Colab Secrets (recommended) ────────────────────────────────────
# from google.colab import userdata
# OPENAI_API_KEY    = userdata.get('OPENAI_API_KEY')
# ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

print('Keys configured.')

## 📄 Cell 3 — Resume & Job Description Data

In [ ]:
import os, re, json, requests
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

SAMPLE_RESUMES = [
    {'name':'Aisha Khan','initials':'AK','role':'ML Engineer','years':6,'color':'#2563EB',
     'resume':('6 years as a Senior ML Engineer. Expert in Python, PyTorch, TensorFlow. '
               'Built production NLP pipelines (BERT, GPT, T5). Deployed LLMs to AWS SageMaker '
               'and GCP Vertex AI. Led a team of 3 engineers. Deep experience with RAG systems, '
               'vector databases (ChromaDB, Pinecone, Weaviate). Strong MLOps: CI/CD, model monitoring. '
               'Published 2 NLP papers. MS Computer Science, Stanford. '
               'Proficient in LangChain, HuggingFace, FastAPI, Docker, Kubernetes.')},
    {'name':'Marco Rivera','initials':'MR','role':'Data Scientist','years':4,'color':'#16A34A',
     'resume':('4 years Data Scientist. Python, scikit-learn, pandas, XGBoost. '
               'NLP: text classification, sentiment analysis using NLTK and spaCy. '
               'No production LLM deployment. SQL, Spark, Airflow. BS Statistics, UCLA.')},
    {'name':'Jamie Park','initials':'JP','role':'AI Researcher','years':3,'color':'#D97706',
     'resume':('3 years AI research, university NLP lab. Expertise in transformers, '
               'fine-tuning LLMs. Published 4 papers. PyTorch, HuggingFace. '
               'RAG prototypes with FAISS. Limited production/cloud experience. PhD candidate, MIT CSAIL.')},
    {'name':'Sofia Li','initials':'SL','role':'NLP Engineer','years':5,'color':'#7C3AED',
     'resume':('5 years NLP engineering. spaCy, HuggingFace Transformers, NLTK. '
               'Built NER, summarization, QA at scale. FastAPI on GCP. '
               'Semantic search, ChromaDB, LangChain. RAG pipelines for enterprise Q&A. BS CS, UT Austin.')},
    {'name':'Devon Walsh','initials':'DW','role':'Software Engineer','years':7,'color':'#0D9488',
     'resume':('7 years software engineering. Python, Java, Go, distributed systems. '
               'Kubernetes, Docker, Terraform, AWS, Azure. No LLM or NLP experience. '
               'Excellent system design. BS Computer Engineering, Georgia Tech.')},
    {'name':'Priya Nair','initials':'PN','role':'MLOps Engineer','years':4,'color':'#C2410C',
     'resume':('4 years MLOps engineering. MLflow, SageMaker, Airflow. '
               'Serving transformer and LLM models in production. LangChain, Pinecone. '
               'MS Software Engineering, Carnegie Mellon.')},
]

DEFAULT_JD = (
    'Senior ML Engineer with 5+ years experience. '
    'Must have: Python, PyTorch, production LLM deployment, NLP expertise, '
    'RAG systems, vector databases (ChromaDB, Pinecone). '
    'Nice to have: MLOps, cloud platforms (AWS/GCP), team leadership. MS/PhD preferred.'
)

print(f'✅ {len(SAMPLE_RESUMES)} sample candidates loaded.')

## 🧠 Cell 4 — RAG Engine

In [ ]:
try:
    from openai import OpenAI as _OpenAI
    _OAI_OK = True
except ImportError:
    _OAI_OK = False

try:
    import anthropic as _ant
    _ANT_OK = True
except ImportError:
    _ANT_OK = False


class HireSenseRAG:


    KEYWORDS = [
        'python','pytorch','tensorflow','llm','nlp','rag','transformer','bert','gpt',
        'embedding','vector','production','deploy','aws','gcp','cloud','mlops','langchain',
        'huggingface','deep learning','machine learning','sql','research','phd','ms',
        'leadership','team','scikit','pandas','api','kubernetes','docker','fine-tun',
        'chromadb','pinecone','spacy','nltk','sagemaker','fastapi','distributed',
        'system design','ci/cd','semantic','faiss','weaviate','airflow','mlflow',
        'monitoring','xgboost','spark',
    ]

    # ── RAG: split_string_into_chunks ──────────────────────────────────
    def split_string_into_chunks(self, text, chunk_size=80):
        words = text.split()
        return [' '.join(words[i:i+chunk_size])
                for i in range(0, len(words), chunk_size)] or [text]

    # ── RAG: model_encode — keyword vector ──────────────────────────────
    def model_encode(self, text):
        lower = text.lower()
        return np.array([
            min(len(re.findall(re.escape(k), lower)) / 2.0, 1.0)
            for k in self.KEYWORDS
        ])

    # ── RAG: cosine_similarity + argmax retrieval ──────────────────────
    def retrieve_top_chunks(self, query_emb, chunks, k=3):
        if not chunks:
            return ''
        sims = cosine_similarity([query_emb],
                                  [self.model_encode(c) for c in chunks])[0]
        top_idx = np.argsort(sims)[::-1][:k]
        return ' '.join(chunks[i] for i in top_idx)

    # ── RAG: full RAG() function ───────────────────────────────────────
    def RAG(self, query, resume_text):
        query_emb = self.model_encode(query)
        chunks    = self.split_string_into_chunks(resume_text, chunk_size=60)
        context   = self.retrieve_top_chunks(query_emb, chunks, k=3)
        similarity = float(cosine_similarity(
            [query_emb], [self.model_encode(resume_text)]
        )[0][0])
        return {'context': context, 'similarity': round(similarity * 100, 1)}

    # ── PromptTemplate pattern ──────────────────────────────────────
    def build_prompt(self, jd, name, context, sim):
        return f"""INSTRUCTIONS:
You are an expert HR talent analyst using a RAG-powered hiring system.
Similarity score: {sim:.1f}%. Analyze ONLY the provided resume context.

Job Description:
{jd}

Candidate: {name}
RAG-Retrieved Context:
{context}

Return ONLY valid JSON (no markdown):
{{"score":<0-100>,"strengths":"<2-3 sentences>","weaknesses":"<2-3 sentences>",
"reasoning":"<2-3 sentences>","skill_match":["s1","s2","s3"],
"skill_gaps":["g1","g2"],"questions":["Q1?","Q2?","Q3?"]}}"""

    # ── prompt_function — OpenAI ──────────────────────────────────────
    def prompt_function_openai(self, prompt, api_key, model='gpt-4o'):
        client = _OpenAI(api_key=api_key)
        r = client.chat.completions.create(
            model=model, temperature=0, max_tokens=1024,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return r.choices[0].message.content

    # ── prompt_function — Anthropic ─────────────────────────────────
    def prompt_function_anthropic(self, prompt, api_key, model='claude-opus-4-5'):
        client = _ant.Anthropic(api_key=api_key)
        m = client.messages.create(
            model=model, max_tokens=1024,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return m.content[0].text

    def prompt_function(self, prompt, provider, api_key, model):
        if provider == 'openai':
            return self.prompt_function_openai(prompt, api_key, model)
        return self.prompt_function_anthropic(prompt, api_key, model)

    # ── Full pipeline: RAG → prompt → LLM ────────────────────────────────────
    def analyze_candidate(self, jd, candidate, provider, api_key, model):
        rag    = self.RAG(jd, candidate['resume'])
        prompt = self.build_prompt(jd, candidate['name'],
                                    rag['context'], rag['similarity'])
        raw    = self.prompt_function(prompt, provider, api_key, model)
        try:
            parsed = json.loads(raw.replace('```json','').replace('```','').strip())
        except Exception:
            parsed = {'score': int(rag['similarity']), 'strengths': 'Strong background.',
                      'weaknesses': 'Some gaps.', 'reasoning': 'Scored by embedding similarity.',
                      'skill_match': [], 'skill_gaps': [], 'questions': []}
        parsed.update({k: candidate[k] for k in ('name','role','years','initials','color')})
        parsed['similarity']  = rag['similarity']
        parsed['rag_context'] = rag['context']
        parsed['provider']    = provider
        return parsed


def fetch_jd_from_url(url):
    """Fetch and extract job description text from a public careers page URL."""
    try:
        proxy = f"https://api.allorigins.win/get?url={requests.utils.quote(url)}"
        data  = requests.get(proxy, timeout=15).json()
        html  = data.get('contents', '')
        text  = re.sub(r'<script[\s\S]*?</script>', '', html, flags=re.I)
        text  = re.sub(r'<style[\s\S]*?</style>',  '', text,  flags=re.I)
        text  = re.sub(r'<[^>]+>', ' ', text)
        text  = re.sub(r'\s{3,}', '\n\n', text).strip()
        for pat in [
            r'(?:job description|about the role|responsibilities)([\s\S]{200,3000}?)(?:about us|benefits|apply)',
            r'(?:qualifications|requirements)([\s\S]{100,2000}?)(?:benefits|about|apply)',
        ]:
            m = re.search(pat, text, re.I)
            if m and len(m.group(0)) > 150:
                return m.group(0).strip()[:3000]
        return text[:2500].strip()
    except Exception as e:
        return f'[Fetch failed: {e}]'


rag_engine = HireSenseRAG()
print(f'✅ HireSenseRAG engine ready.')
print(f'   OpenAI available:    {_OAI_OK}')
print(f'   Anthropic available: {_ANT_OK}')


## 🎛️ Cell 5 — Interactive Dashboard

**This is the main UI.** Run to launch the full recruiter dashboard with:
- PDF/TXT resume upload
- Paste or URL-fetch job descriptions
- ChatGPT, Claude, or both side-by-side
- Ranked candidates with scores, skill tags, reasoning, interview questions

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── CSS ───────────────────────────────────────────────────────────────────────
display(HTML("""
<style>
body, .hs-wrap { font-family: -apple-system,BlinkMacSystemFont,"Inter",sans-serif; }
.hs-header { background:#0F1F3D; border-radius:12px; padding:18px 22px;
             margin-bottom:14px; display:flex; align-items:center; gap:14px; }
.hs-logo   { width:38px; height:38px; background:linear-gradient(135deg,#2563EB,#7C3AED);
             border-radius:9px; display:flex; align-items:center;
             justify-content:center; font-size:20px; flex-shrink:0; }
.hs-title  { font-size:20px; font-weight:700; color:#FFF; }
.hs-sub    { font-size:11px; color:#93C5FD; letter-spacing:.06em; margin-top:2px; }
.hs-badges { display:flex; gap:8px; flex-wrap:wrap; margin-left:auto; }
.hs-badge  { font-size:11px; padding:3px 10px; border-radius:20px; font-weight:600; }
.hs-card   { background:#FFF; border:1.5px solid #DDE3EE; border-radius:12px;
             padding:16px; margin-bottom:12px; box-shadow:0 1px 4px rgba(15,31,61,.07); }
.hs-slabel { font-size:10px; color:#6B7A99; letter-spacing:.07em; font-weight:600;
             text-transform:uppercase; margin-bottom:8px; }
.hs-metrics { display:grid; grid-template-columns:repeat(4,1fr); gap:10px; margin-bottom:14px; }
.hs-metric  { background:#FFF; border:1.5px solid #DDE3EE; border-radius:10px;
              padding:14px; text-align:center; box-shadow:0 1px 3px rgba(15,31,61,.06); }
.hs-mval   { font-size:26px; font-weight:700; line-height:1; }
.hs-mlbl   { font-size:10px; color:#6B7A99; margin-top:4px; font-weight:500; }
.hs-ccard  { background:#FFF; border:1.5px solid #DDE3EE; border-radius:12px;
             padding:16px; margin-bottom:10px; box-shadow:0 1px 3px rgba(15,31,61,.06); }
.hs-crow   { display:flex; align-items:center; gap:12px; }
.hs-avatar { width:42px; height:42px; border-radius:50%; display:flex;
             align-items:center; justify-content:center; font-weight:700;
             font-size:14px; flex-shrink:0; }
.hs-cname  { font-size:14px; font-weight:700; color:#0F1F3D; }
.hs-crole  { font-size:11px; color:#6B7A99; margin-top:2px; }
.hs-score  { margin-left:auto; text-align:right; }
.hs-snum   { font-size:28px; font-weight:700; line-height:1; }
.hs-sbar   { width:80px; height:4px; background:#DDE3EE; border-radius:2px;
             margin-top:5px; margin-left:auto; overflow:hidden; }
.hs-sfill  { height:4px; border-radius:2px; }
.hs-detail { background:#F7FAFF; border-radius:8px; padding:14px;
             margin-top:12px; border-top:1.5px solid #DDE3EE; }
.hs-2col   { display:grid; grid-template-columns:1fr 1fr; gap:16px; margin-bottom:12px; }
.hs-sh4    { font-size:9px; color:#6B7A99; letter-spacing:.07em; font-weight:600;
             text-transform:uppercase; margin-bottom:5px; }
.hs-sp     { font-size:12px; color:#374151; line-height:1.55; }
.hs-reason { background:#FFF; border-radius:7px; padding:10px 12px;
             margin-bottom:12px; font-size:12px; color:#374151; line-height:1.55; }
.hs-tags   { display:flex; flex-wrap:wrap; gap:5px; margin-bottom:12px; }
.hs-tag    { font-size:11px; padding:3px 9px; border-radius:20px; font-weight:500; }
.hs-qrow   { display:flex; align-items:flex-start; gap:8px; margin-bottom:8px; }
.hs-qnum   { font-size:10px; font-weight:700; padding:2px 7px; border-radius:20px;
             flex-shrink:0; margin-top:1px; }
.hs-qtext  { font-size:12px; color:#374151; line-height:1.4; }
.hs-ptag   { font-size:10px; padding:2px 8px; border-radius:20px; font-weight:600; }
.hs-rag    { background:#0F1F3D; border-radius:7px; padding:10px; font-size:10px;
             color:#93C5FD; font-family:monospace; line-height:1.5;
             max-height:80px; overflow-y:auto; white-space:pre-wrap; margin-top:6px; }
.hs-pipe   { background:#FFF; border:1.5px solid #DDE3EE; border-radius:8px;
             padding:10px 14px; display:flex; align-items:center; gap:8px;
             flex-wrap:wrap; font-size:11px; color:#6B7A99; margin-bottom:14px; font-weight:500; }
.hs-pdot   { width:8px; height:8px; border-radius:50%; background:#16A34A; flex-shrink:0; }
.cmp-tbl   { width:100%; border-collapse:collapse; margin-top:10px; font-size:13px; }
.cmp-tbl th { font-size:10px; color:#6B7A99; letter-spacing:.06em; padding:9px 12px;
              border-bottom:2px solid #DDE3EE; font-weight:600; text-align:center; }
.cmp-tbl th:first-child { text-align:left; }
.cmp-tbl td { padding:9px 12px; border-bottom:1px solid #DDE3EE; text-align:center; }
.cmp-tbl td:first-child { text-align:left; color:#0F1F3D; font-weight:600; }
</style>
"""))

display(HTML("""
<div class="hs-wrap">
<div class="hs-header">
  <div class="hs-logo">⚡</div>
  <div>
    <div class="hs-title">HireSense AI</div>
    <div class="hs-sub">RAG-POWERED TALENT INTELLIGENCE</div>
  </div>
  <div class="hs-badges">
    <span class="hs-badge" style="background:rgba(37,99,235,.15);color:#2563EB">RAG Pipeline</span>
    <span class="hs-badge" style="background:rgba(13,148,136,.15);color:#0D9488">Cosine Similarity</span>
    <span class="hs-badge" style="background:rgba(124,58,237,.15);color:#7C3AED">LLM Scoring</span>
    <span class="hs-badge" style="background:rgba(5,150,105,.15);color:#059669">OpenAI</span>
    <span class="hs-badge" style="background:rgba(194,65,12,.15);color:#C2410C">Claude</span>
  </div>
</div>
</div>
"""))

W = dict(layout=widgets.Layout(width='100%'))

# ── Provider ──────────────────────────────────────────────────────────────────
provider_w = widgets.ToggleButtons(
    options=[('🟢  ChatGPT (OpenAI)','openai'),
             ('🟠  Claude (Anthropic)','anthropic'),
             ('⚡  Both','both')],
    value='openai',
    style={'button_width':'190px','description_width':'0'},
)
oai_key_w   = widgets.Password(placeholder='sk-...',
    description='OpenAI Key:', style={'description_width':'110px'}, **W)
oai_model_w = widgets.Dropdown(
    options=['gpt-4o','gpt-4o-mini','gpt-4-turbo','gpt-3.5-turbo'],
    value='gpt-4o', description='Model:', style={'description_width':'110px'}, **W)
ant_key_w   = widgets.Password(placeholder='sk-ant-...',
    description='Anthropic Key:', style={'description_width':'110px'},
    layout=widgets.Layout(width='100%', display='none'))
ant_model_w = widgets.Dropdown(
    options=['claude-opus-4-5','claude-sonnet-4-5','claude-haiku-4-5-20251001'],
    value='claude-opus-4-5', description='Model:',
    style={'description_width':'110px'},
    layout=widgets.Layout(width='100%', display='none'))

if OPENAI_API_KEY:    oai_key_w.value = OPENAI_API_KEY
if ANTHROPIC_API_KEY: ant_key_w.value = ANTHROPIC_API_KEY

def on_provider(change):
    p = change['new']
    oai_key_w.layout.display    = 'none' if p == 'anthropic' else ''
    oai_model_w.layout.display  = 'none' if p == 'anthropic' else ''
    ant_key_w.layout.display    = 'none' if p == 'openai'    else ''
    ant_model_w.layout.display  = 'none' if p == 'openai'    else ''
provider_w.observe(on_provider, names='value')

# ── Job Description ───────────────────────────────────────────────────────────
jd_text_w    = widgets.Textarea(value=DEFAULT_JD,
    placeholder='Paste job description...', rows=6, **W)
jd_url_w     = widgets.Text(placeholder='https://careers.company.com/job/...',
    description='URL:', style={'description_width':'40px'}, **W)
jd_fetched_w = widgets.Textarea(
    placeholder='Fetched text appears here — edit freely...', rows=6, **W)
fetch_btn_w  = widgets.Button(description='Fetch JD', button_style='info',
                               layout=widgets.Layout(width='110px'))
fetch_out_w  = widgets.Output()

def on_fetch(_):
    with fetch_out_w:
        clear_output()
        url = jd_url_w.value.strip()
        if not url: print('⚠ Enter a URL first.'); return
        print(f'Fetching {url[:55]}...')
        text = fetch_jd_from_url(url)
        jd_fetched_w.value = text
        print(f'✓ {len(text)} characters extracted.')
fetch_btn_w.on_click(on_fetch)

jd_tabs = widgets.Tab(children=[
    widgets.VBox([jd_text_w]),
    widgets.VBox([widgets.HBox([jd_url_w, fetch_btn_w]), fetch_out_w, jd_fetched_w]),
])
jd_tabs.set_title(0, '📋 Paste')
jd_tabs.set_title(1, '🌐 From URL')

# ── File upload ───────────────────────────────────────────────────────────────
upload_w     = widgets.FileUpload(accept='.pdf,.txt', multiple=True,
                                   description='Upload Resumes',
                                   layout=widgets.Layout(width='100%'))
upload_out_w = widgets.Output()
extra_candidates = []
_ucidx = [0]
UPLOAD_COLORS = ['#7C3AED','#DC2626','#D97706','#059669','#2563EB','#DB2777']

def on_upload(change):
    with upload_out_w:
        clear_output()
        for fname, fdata in upload_w.value.items():
            raw = fdata['content']
            ext = fname.split('.')[-1].lower()
            text = ''
            if ext == 'txt':
                text = raw.decode('utf-8', errors='replace')
            elif ext == 'pdf':
                try:
                    s = raw.decode('latin-1', errors='replace')
                    for blk in re.findall(r'BT[\s\S]*?ET', s):
                        for m in re.findall(r'\(([^)]{1,300})\)\s*(?:Tj|TJ)', blk):
                            text += m + ' '
                    if len(text.strip()) < 50:
                        text = ' '.join(re.findall(
                            r'\(([A-Za-z0-9 ,.\'\"-:;@+&/]{4,200})\)', s))
                except Exception:
                    text = f'[Could not parse {fname}]'
            if not text.strip():
                print(f'⚠ No text from {fname}'); continue
            name  = fname.replace('.pdf','').replace('.txt','').replace('-',' ').replace('_',' ').title()
            ini   = ''.join(w[0] for w in name.split()[:2]).upper()
            color = UPLOAD_COLORS[_ucidx[0] % len(UPLOAD_COLORS)]
            _ucidx[0] += 1
            extra_candidates.append({'name':name,'initials':ini,'role':'Uploaded',
                                      'years':None,'color':color,'resume':text})
            print(f'✅ Uploaded: {name} ({len(text)} chars)')
upload_w.observe(on_upload, names='value')

# ── Candidate checkboxes ──────────────────────────────────────────────────────
cand_checks = [
    widgets.Checkbox(value=True,
        description=f"{r['name']} — {r['role']} ({r['years']}y)",
        style={'description_width':'initial'})
    for r in SAMPLE_RESUMES
]

run_btn_w   = widgets.Button(description='▶  Run HireSense Analysis',
                              button_style='primary',
                              layout=widgets.Layout(width='100%', height='42px'))
status_out  = widgets.Output()
results_out = widgets.Output()

# ── HTML helpers ──────────────────────────────────────────────────────────────
AV_COLS   = ['#2563EB','#16A34A','#D97706','#7C3AED','#0D9488','#C2410C']
RANK_ICONS = ['🥇','🥈','🥉','4','5','6','7','8']

def sc(s):
    return '#16A34A' if s>=75 else '#D97706' if s>=50 else '#DC2626'

def cand_card_html(r, rank, col):
    ini  = r.get('initials', r['name'][:2].upper())
    s    = sc(r['score'])
    prov = r.get('provider','')
    pbg  = '#ECFDF5' if prov=='openai' else '#FFF7ED'
    pc   = '#059669' if prov=='openai' else '#C2410C'
    pl   = 'ChatGPT' if prov=='openai' else 'Claude' if prov else ''
    ptag = (f"<span class='hs-ptag' style='background:{pbg};color:{pc};'>{pl}</span>"
            if pl else '')
    skill_tags = ''.join(
        f"<span class='hs-tag' style='background:#F0FDF4;color:#16A34A;'>{x}</span>"
        for x in r.get('skill_match',[]))
    gap_tags = ''.join(
        f"<span class='hs-tag' style='background:#FEF2F2;color:#DC2626;'>{x}</span>"
        for x in r.get('skill_gaps',[]))
    qs = ''.join(
        f"<div class='hs-qrow'><span class='hs-qnum' style='background:#EFF6FF;color:#2563EB;'>Q{i+1}</span>"
        f"<span class='hs-qtext'>{q}</span></div>"
        for i,q in enumerate(r.get('questions',[])))
    rag = r.get('rag_context','').replace('<','&lt;')[:400]
    return f"""
    <div class="hs-ccard">
      <div class="hs-crow">
        <div style="width:28px;height:28px;border-radius:50%;background:#EFF6FF;color:#2563EB;
                    display:flex;align-items:center;justify-content:center;
                    font-weight:700;font-size:12px;flex-shrink:0">{RANK_ICONS[rank]}</div>
        <div class="hs-avatar" style="background:{col}22;color:{col};border:2px solid {col}">{ini}</div>
        <div style="flex:1">
          <div class="hs-cname">{r['name']} {ptag}</div>
          <div class="hs-crole">{r.get('role','Uploaded')} &middot; sim {r['similarity']}%</div>
        </div>
        <div class="hs-score">
          <div class="hs-snum" style="color:{s}">{r['score']}</div>
          <div style="font-size:10px;color:#6B7A99">/ 100</div>
          <div class="hs-sbar"><div class="hs-sfill" style="width:{r['score']}%;background:{s}"></div></div>
        </div>
      </div>
      <div class="hs-detail">
        <div class="hs-2col">
          <div><div class="hs-sh4">Strengths</div><div class="hs-sp">{r.get('strengths','—')}</div></div>
          <div><div class="hs-sh4">Gaps</div><div class="hs-sp">{r.get('weaknesses','—')}</div></div>
        </div>
        <div class="hs-sh4">Reasoning</div>
        <div class="hs-reason" style="border-left:3px solid {col}">{r.get('reasoning','—')}</div>
        {('<div class="hs-sh4">Matching Skills</div><div class="hs-tags">' + skill_tags + '</div>') if skill_tags else ''}
        {('<div class="hs-sh4">Skill Gaps</div><div class="hs-tags">' + gap_tags + '</div>') if gap_tags else ''}
        <div class="hs-sh4">Interview Questions</div>
        {qs if qs else '<span style="color:#6B7A99;font-size:12px">—</span>'}
        {('<details style="margin-top:10px"><summary style="font-size:10px;color:#6B7A99;cursor:pointer">▶ RAG CONTEXT</summary><div class="hs-rag">' + rag + '</div></details>') if rag else ''}
      </div>
    </div>"""

def metrics_html(results):
    top  = results[0]
    avgs = round(sum(r['score'] for r in results)/len(results))
    avsm = round(sum(r['similarity'] for r in results)/len(results),1)
    s    = sc(top['score'])
    return f"""<div class="hs-metrics">
      <div class="hs-metric"><div class="hs-mval" style="color:#2563EB">{len(results)}</div><div class="hs-mlbl">Candidates</div></div>
      <div class="hs-metric"><div class="hs-mval" style="color:{s}">{top['score']}</div><div class="hs-mlbl">Top Score</div></div>
      <div class="hs-metric"><div class="hs-mval">{avgs}</div><div class="hs-mlbl">Avg Score</div></div>
      <div class="hs-metric"><div class="hs-mval" style="color:#7C3AED">{avsm}%</div><div class="hs-mlbl">Avg Similarity</div></div>
    </div>"""

def pipe_bar_html():
    steps = ['Chunk','Embed','Cosine Sim','RAG Retrieve','LLM Score','Done ✓']
    inner = ' <span style="color:#DDE3EE">→</span> '.join(
        f"<span class='hs-pdot'></span> {s}" for s in steps)
    return (f"<div class='hs-pipe'>{inner}"
            "<span style='margin-left:auto;color:#16A34A;font-weight:600'>Complete ✓</span></div>")

def compare_html(oai, ant):
    om = {r['name']:r['score'] for r in oai}
    am = {r['name']:r['score'] for r in ant}
    names = list({**om,**am})
    rows = ''
    for n in names:
        o,a = om.get(n,0), am.get(n,0)
        d   = o-a
        dc  = '#059669' if d>0 else '#C2410C' if d<0 else '#6B7A99'
        rows += (f"<tr><td>{n}</td>"
                 f"<td style='color:#059669;font-weight:700'>{o}</td>"
                 f"<td style='color:#C2410C;font-weight:700'>{a}</td>"
                 f"<td style='color:{dc};font-weight:700'>{d:+d}</td></tr>")
    return f"""<div class="hs-card"><div class="hs-slabel">Score Comparison — ChatGPT vs Claude</div>
    <table class="cmp-tbl"><thead><tr>
      <th style="text-align:left">Candidate</th>
      <th style="color:#059669">ChatGPT</th>
      <th style="color:#C2410C">Claude</th>
      <th>Δ Diff</th>
    </tr></thead><tbody>{rows}</tbody></table></div>"""

# ── Run callback ──────────────────────────────────────────────────────────────
def on_run(_):
    with status_out:  clear_output()
    with results_out: clear_output()

    p   = provider_w.value
    jd  = (jd_text_w.value if jd_tabs.selected_index == 0
           else jd_fetched_w.value).strip()
    ok  = oai_key_w.value.strip()
    ak  = ant_key_w.value.strip()
    om  = oai_model_w.value
    am  = ant_model_w.value

    sel_s = [SAMPLE_RESUMES[i] for i,c in enumerate(cand_checks) if c.value]
    all_c = sel_s + extra_candidates

    with status_out:
        if not jd:       print('⚠ Please enter a job description.'); return
        if not all_c:    print('⚠ Select or upload at least one candidate.'); return
        if p in ('openai','both')     and not ok: print('⚠ OpenAI API key required.');    return
        if p in ('anthropic','both')  and not ak: print('⚠ Anthropic API key required.'); return
        print('⏳ Running RAG pipeline...')

    provs = ['openai','anthropic'] if p == 'both' else [p]
    all_results = {}

    for prov in provs:
        key = ok if prov=='openai' else ak
        mod = om if prov=='openai' else am
        results = []
        for cand in all_c:
            with status_out:
                clear_output(wait=True)
                print(f'⏳ [{prov}] Analyzing {cand["name"]}...')
            try:
                r = rag_engine.analyze_candidate(jd, cand, prov, key, mod)
            except Exception as e:
                r = {'name':cand['name'],'role':cand.get('role','Uploaded'),
                     'years':cand.get('years'),'initials':cand.get('initials','??'),
                     'color':cand.get('color','#2563EB'),
                     'score':0,'similarity':0.0,'strengths':'Error',
                     'weaknesses':str(e)[:200],'reasoning':'API call failed.',
                     'skill_match':[],'skill_gaps':[],'questions':[],'rag_context':'','provider':prov}
            results.append(r)
        results.sort(key=lambda x: x['score'], reverse=True)
        all_results[prov] = results

    with status_out:
        clear_output()
        print(f'✅ Analyzed {len(all_c)} candidate(s) via {" + ".join(provs)}')

    with results_out:
        clear_output()
        if p == 'both':
            display(HTML(compare_html(all_results['openai'], all_results['anthropic'])))
            display(HTML('<br>'))
            for prov_key, prov_lbl, prov_col in [
                ('openai',    'ChatGPT (OpenAI)',   '#059669'),
                ('anthropic', 'Claude (Anthropic)', '#C2410C'),
            ]:
                res = all_results[prov_key]
                display(HTML(
                    f"<div style='background:{prov_col}12;border:1px solid {prov_col}33;"
                    f"border-radius:10px;padding:12px 16px;margin-bottom:8px'>"
                    f"<div style='font-size:13px;font-weight:700;color:{prov_col};"
                    f"margin-bottom:10px'>● {prov_lbl}</div>"
                    f"{metrics_html(res)}</div>"
                ))
                for rank, r in enumerate(res):
                    display(HTML(cand_card_html(r, rank, AV_COLS[rank % len(AV_COLS)])))
        else:
            res = all_results[provs[0]]
            display(HTML(metrics_html(res)))
            display(HTML(pipe_bar_html()))
            for rank, r in enumerate(res):
                display(HTML(cand_card_html(r, rank, AV_COLS[rank % len(AV_COLS)])))

run_btn_w.on_click(on_run)

# ── Render layout ─────────────────────────────────────────────────────────────
display(HTML("<div class='hs-card'><div class='hs-slabel'>LLM Provider</div>"))
display(provider_w, oai_key_w, oai_model_w, ant_key_w, ant_model_w)
display(HTML("</div>"))

display(HTML("<div class='hs-card'><div class='hs-slabel'>Job Description</div>"))
display(jd_tabs)
display(HTML("</div>"))

display(HTML("<div class='hs-card'><div class='hs-slabel'>Upload Resumes (PDF or TXT)</div>"))
display(upload_w, upload_out_w)
display(HTML("</div>"))

display(HTML("<div class='hs-card'><div class='hs-slabel'>Sample Candidates</div>"))
for chk in cand_checks:
    display(chk)
display(HTML("</div>"))

display(run_btn_w)
display(status_out)
display(HTML('<br>'))
display(results_out)


## 🔬 Cell 6 — Inspect RAG Retrieval

See exactly which resume chunks are retrieved for any candidate.

In [ ]:
# Change CANDIDATE_INDEX (0=Aisha, 1=Marco, 2=Jamie, 3=Sofia, 4=Devon, 5=Priya)
JD_QUERY        = DEFAULT_JD
CANDIDATE_INDEX = 0

cand   = SAMPLE_RESUMES[CANDIDATE_INDEX]
result = rag_engine.RAG(JD_QUERY, cand['resume'])

print(f'Candidate:  {cand["name"]} ({cand["role"]})')
print(f'Similarity: {result["similarity"]}%')
print(f'\nRAG-Retrieved Context (top 3 chunks by cosine similarity):')
print('-' * 60)
print(result['context'])

## 📊 Cell 7 — Text Summary Table (optional)

Run after Cell 5 to see a clean ranked table.

In [ ]:
# Run after Cell 5
try:
    for prov, res in all_results.items():
        label = 'ChatGPT (OpenAI)' if prov=='openai' else 'Claude (Anthropic)'
        print(f'\n{"="*58}')
        print(f'  {label}')
        print(f'{"="*58}')
        print(f'  {"Rank":<5} {"Name":<20} {"Role":<20} {"Score":>5} {"Sim":>7}')
        print(f'  {"-"*5} {"-"*20} {"-"*20} {"-"*5} {"-"*7}')
        for i, r in enumerate(res):
            print(f'  {i+1:<5} {r["name"]:<20} {r["role"]:<20} {r["score"]:>5} {r["similarity"]:>6}%')
except NameError:
    print('Run Cell 5 first.')